# Document-level CID with PubMedBERT / BioBERT (Part 11)

Fine-tunes a biomedical transformer with entity markers on the same CDR split as
`zheng2018_cid.ipynb` Part 10. Scores match Part 7 (gold includes CIDs beyond K).

**Setup:** PyTorch + Hugging Face — run in a **separate kernel** from the TensorFlow notebook
so they do not share the 6 GB GPU.

Requires `results/prepared_docs.pkl`, `results/split.json` (from Part 10 Cell 10.0), and
`results/mesh_tree_numbers.json`.


In [ ]:
# PyTorch with CUDA + Hugging Face transformers, in THIS kernel's Python.
import importlib.util, subprocess, sys
missing = [p for p, m in [("torch", "torch"), ("transformers", "transformers")]
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("installed now:", missing or "nothing")


In [ ]:
import json, pickle, random, gc, math, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

REPO_ROOT = Path(".").resolve()
RESULTS_DIR = REPO_ROOT / "results"

MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"   # PubMedBERT
# MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"                             # BioBERT
SHORT_NAME = "pubmedbert"        # used in the result keys: change to "biobert" with BioBERT

MAX_LEN      = 512
BATCH_SIZE   = 2                 # 6 GB GPU
GRAD_ACCUM   = 4                 # 2 x 4 = 8 pairs per update (the paper's batch size)
LR           = 2e-5
EPOCHS       = 5
PATIENCE     = 2                 # stop after 2 epochs without a better dev F1
THRESHOLD    = 0.5
K_WINDOW     = 4
SEEDS        = [1, 2, 3]
GRADIENT_CHECKPOINTING = True    # less GPU memory, ~30% slower

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = DEVICE.type == "cuda"
print("device:", DEVICE, torch.cuda.get_device_name(0) if USE_FP16 else "")


In [ ]:
with open(RESULTS_DIR / "prepared_docs.pkl", "rb") as f:
    train_docs, dev_docs, test_docs = pickle.load(f)
split = json.loads((RESULTS_DIR / "split.json").read_text())
by_pmid = {d["pmid"]: d for d in train_docs + dev_docs + test_docs}
new_train_docs = [by_pmid[p] for p in split["train"]]
new_dev_docs = [by_pmid[p] for p in split["dev"]]
test_docs = [by_pmid[p] for p in split["test"]]
tree_numbers = json.loads((RESULTS_DIR / "mesh_tree_numbers.json").read_text())

gold_dev = {(d["pmid"], c, s) for d in new_dev_docs for (c, s) in d["cids"]}
gold = {(d["pmid"], c, s) for d in test_docs for (c, s) in d["cids"]}


def candidate_pairs(doc, k_limit=None):
    """Same as Part 3: (chem, dis, label, span)."""
    chems = {i for e in doc["entities"] if e["type"] == "Chemical" for i in e["ids"]}
    dises = {i for e in doc["entities"] if e["type"] == "Disease" for i in e["ids"]}
    pairs = []
    for c in sorted(chems):
        for d in sorted(dises):
            if not doc["sent_of"].get(c) or not doc["sent_of"].get(d):
                continue
            span = min(abs(i - j) for i in doc["sent_of"][c] for j in doc["sent_of"][d]) + 1
            if k_limit is not None and span > k_limit:
                continue
            pairs.append((c, d, int((c, d) in doc["cids"]), span))
    return pairs


def prf(pred, gold_set):
    tp = len(pred & gold_set)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold_set) if gold_set else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return round(100 * p, 1), round(100 * r, 1), round(100 * f, 1)


def is_intra(pmid, c, d):
    doc = by_pmid[pmid]
    return len(set(doc["sent_of"][c]) & set(doc["sent_of"][d])) > 0


def rule_1(pred, docs):
    new = set(pred)
    found = {pmid for pmid, _, _ in pred}
    for doc in docs:
        if doc["pmid"] in found:
            continue
        chem = Counter(i for e in doc["entities"] if e["type"] == "Chemical" for i in e["ids"])
        dis = Counter(i for e in doc["entities"] if e["type"] == "Disease" for i in e["ids"])
        if chem and dis:
            new.add((doc["pmid"], chem.most_common(1)[0][0], dis.most_common(1)[0][0]))
    return new


def more_general(a, b):
    return any(tb.startswith(ta + ".") for ta in tree_numbers.get(a, []) for tb in tree_numbers.get(b, []))


def rule_2(pred, docs):
    new = set(pred)
    for doc in docs:
        diseases = {i for e in doc["entities"] if e["type"] == "Disease" for i in e["ids"]}
        in_title = {i for e in doc["entities"]
                    if e["type"] == "Disease" and e["start"] < len(doc["title"]) for i in e["ids"]}
        for d in diseases - in_title:
            if any(more_general(d, o) for o in diseases if o != d):
                new = {x for x in new if not (x[0] == doc["pmid"] and x[2] == d)}
    return new


RESULTS_FILE = RESULTS_DIR / "results_bert.json"   # separate file: two notebooks never write the same one
RESULTS = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.is_file() else {}


def save_result(key, value):
    RESULTS[key] = value
    RESULTS_FILE.write_text(json.dumps(RESULTS, indent=2))


print(len(new_train_docs), "train /", len(new_dev_docs), "dev /", len(test_docs), "test articles")


In [ ]:
MARKERS = ["[CHEM]", "[/CHEM]", "[DIS]", "[/DIS]"]


def marked_text(doc, chem, dis):
    """Title + abstract, with every mention of the candidate pair wrapped in markers."""
    text = doc["title"] + " " + doc["abstract"]
    parts, cursor = [], 0
    for e in sorted(doc["entities"], key=lambda e: e["start"]):
        if e["start"] < cursor:
            continue                                       # overlapping mention
        if e["type"] == "Chemical" and chem in e["ids"]:
            left, right = " [CHEM] ", " [/CHEM] "
        elif e["type"] == "Disease" and dis in e["ids"]:
            left, right = " [DIS] ", " [/DIS] "
        else:
            continue
        parts += [text[cursor:e["start"]], left, text[e["start"]:e["end"]], right]
        cursor = e["end"]
    parts.append(text[cursor:])
    return "".join(parts)


d0 = new_train_docs[0]
c0, s0, _, _ = candidate_pairs(d0)[0]
print(marked_text(d0, c0, s0)[:400])


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": MARKERS})
CHEM_ID, DIS_ID = tokenizer.convert_tokens_to_ids(["[CHEM]", "[DIS]"])


def make_examples(docs, k_limit):
    texts, examples = [], []
    for doc in docs:
        for c, d, label, span in candidate_pairs(doc, k_limit):
            texts.append(marked_text(doc, c, d))
            examples.append({"pmid": doc["pmid"], "chem": c, "dis": d, "label": label})
    encoded = tokenizer(texts, truncation=True, max_length=MAX_LEN)["input_ids"]
    lost = 0
    for ex, ids in zip(examples, encoded):
        ex["input_ids"] = ids
        lost += (CHEM_ID not in ids) or (DIS_ID not in ids)
    return examples, lost


train_ex, lost_tr = make_examples(new_train_docs, None)    # no K filter for training, as in the paper
dev_ex, lost_dv = make_examples(new_dev_docs, K_WINDOW)
test_ex, lost_te = make_examples(test_docs, K_WINDOW)
lengths = [len(e["input_ids"]) for e in train_ex]
print(f"pairs: train {len(train_ex)}, dev {len(dev_ex)}, test {len(test_ex)}")
print(f"tokens per pair: median {int(np.median(lengths))}, max {max(lengths)}; "
      f"cut at {MAX_LEN}: {sum(l >= MAX_LEN for l in lengths)} train pairs")
print(f"pairs where a marked entity was cut off: train {lost_tr}, dev {lost_dv}, test {lost_te}")


In [ ]:
def batches(examples, batch_size, shuffle):
    """Pad each batch to its own longest pair (fast: most abstracts are far below 512 tokens)."""
    order = list(range(len(examples)))
    if shuffle:
        random.shuffle(order)
    for i in range(0, len(order), batch_size):
        chunk = [examples[j] for j in order[i:i + batch_size]]
        length = max(len(e["input_ids"]) for e in chunk)
        ids = torch.full((len(chunk), length), tokenizer.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(chunk), length), dtype=torch.long)
        for k, e in enumerate(chunk):
            ids[k, :len(e["input_ids"])] = torch.tensor(e["input_ids"])
            mask[k, :len(e["input_ids"])] = 1
        labels = torch.tensor([e["label"] for e in chunk], dtype=torch.long)
        yield ids, mask, labels


In [ ]:
class MarkerClassifier(nn.Module):
    """The transformer reads the marked abstract. Pair vector =
    [CLS] || mean state at [CHEM] || mean state at [DIS], then a small classifier."""

    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.encoder.resize_token_embeddings(len(tokenizer))           # room for the 4 markers
        if GRADIENT_CHECKPOINTING:
            self.encoder.gradient_checkpointing_enable()
        h = self.encoder.config.hidden_size                            # 768
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(3 * h, h), nn.ReLU(),
                                  nn.Dropout(0.1), nn.Linear(h, 2))

    @staticmethod
    def pool(states, ids, token_id):
        """Mean of the states at every occurrence of token_id (0 if it was cut off)."""
        m = (ids == token_id).unsqueeze(-1).to(states.dtype)
        return (states * m).sum(1) / m.sum(1).clamp(min=1)

    def forward(self, ids, mask):
        states = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        pair = torch.cat([states[:, 0],
                          self.pool(states, ids, CHEM_ID),
                          self.pool(states, ids, DIS_ID)], dim=-1)
        return self.head(pair)                                          # logits: [not CID, CID]


In [ ]:
def predict(model, examples):
    model.eval()
    probs = []
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_FP16):
        for ids, mask, _ in batches(examples, 16, shuffle=False):
            logits = model(ids.to(DEVICE), mask.to(DEVICE))
            probs += torch.softmax(logits.float(), dim=-1)[:, 1].tolist()
    return np.array(probs)


def to_pairs(examples, probs, thr=THRESHOLD):
    return {(e["pmid"], e["chem"], e["dis"]) for e, p in zip(examples, probs) if p >= thr}


def train_one(seed):
    """Fine-tune one model; keep the epoch with the best dev F1 (early stopping)."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = MarkerClassifier().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    updates = EPOCHS * math.ceil(len(train_ex) / (BATCH_SIZE * GRAD_ACCUM))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * updates), updates)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)
    loss_fn = nn.CrossEntropyLoss()

    best_f, best_state, bad = -1.0, None, 0
    for epoch in range(EPOCHS):
        model.train()
        start, total = time.time(), 0.0
        optimizer.zero_grad()
        for step, (ids, mask, y) in enumerate(batches(train_ex, BATCH_SIZE, shuffle=True)):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_FP16):
                loss = loss_fn(model(ids.to(DEVICE), mask.to(DEVICE)).float(), y.to(DEVICE)) / GRAD_ACCUM
            scaler.scale(loss).backward()
            total += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()

        p, r, f = prf(to_pairs(dev_ex, predict(model, dev_ex)), gold_dev)
        print(f"  epoch {epoch + 1}: loss {total / (step + 1):.4f} | dev P/R/F = {p} / {r} / {f} "
              f"| {time.time() - start:.0f} s")
        if f > best_f:
            best_f, bad = f, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stopping")
                break

    model.load_state_dict(best_state)
    print(f"  best dev F1 {best_f}")
    return model


In [ ]:
for seed in SEEDS:
    key = f"{SHORT_NAME}_s{seed}"
    if key in RESULTS:
        print(key, "already done:", RESULTS[key]["no_rules"])
        continue
    print(f"\n=== {key} ===")
    model = train_one(seed)
    probs = predict(model, test_ex)
    pred = to_pairs(test_ex, probs)
    save_result(key, {"no_rules": prf(pred, gold),
                      "rules": prf(rule_2(rule_1(pred, test_docs), test_docs), gold),
                      "pred": sorted(pred)})
    print(key, "test P / R / F:", RESULTS[key]["no_rules"], "| with rules:", RESULTS[key]["rules"])
    del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
def mean_std(rows):
    a = np.array(rows, dtype=float)
    return " / ".join(f"{m:.1f} ± {s:.1f}" for m, s in zip(a.mean(axis=0), a.std(axis=0)))


def intra_inter(pred):
    out = []
    for wanted in (True, False):
        g = {x for x in gold if is_intra(*x) == wanted}
        p = {x for x in pred if is_intra(*x) == wanted}
        out.append(prf(p, g)[2])
    return out


rows = []
done = [RESULTS[k] for k in RESULTS if k.startswith(SHORT_NAME + "_s")]
if done:
    preds = [{tuple(x) for x in r["pred"]} for r in done]
    rows.append((SHORT_NAME, len(done), mean_std([r["no_rules"] for r in done]),
                 mean_std([r["rules"] for r in done]), mean_std([intra_inter(p) for p in preds])))

# Part 10 baseline (predictions written by the other notebook)
base_preds = [{tuple(x) for x in json.loads((RESULTS_DIR / f"pred_base_s{s}.json").read_text())}
              for s in SEEDS if (RESULTS_DIR / f"pred_base_s{s}.json").is_file()]
if base_preds:
    rows.append(("LSTM+CNN baseline", len(base_preds), mean_std([prf(p, gold) for p in base_preds]),
                 mean_std([prf(rule_2(rule_1(p, test_docs), test_docs), gold) for p in base_preds]),
                 mean_std([intra_inter(p) for p in base_preds])))

print(f"{'Model':<20}{'seeds':>6}   {'P / R / F (no rules)':<40}{'P / R / F (rules)':<40}{'F intra / inter'}")
for name, n, a, b, c in rows:
    print(f"{name:<20}{n:>6}   {a:<40}{b:<40}{c}")
print("\nPaper: 54.3 / 65.9 / 59.5 (no rules) · 56.2 / 68.0 / 61.5 (rules) · intra 65.3 / inter 44.0")


### If the GPU runs out of memory (`CUDA out of memory`)

Try these in order, re-running from the settings cell:
1. `BATCH_SIZE = 1` and `GRAD_ACCUM = 8` (same effective batch).
2. `MAX_LEN = 384` (check the "cut off" counts afterwards).
3. Close the TensorFlow notebook (or restart its kernel): it may still hold GPU memory.
